In [ ]:
import json
import pandas as pd

from Pipeline.Model.GallstoneDataSet import GallstoneDataSet
from Pipeline.Model.ExtremeLearningMachine import ExtremeLearningMachine, sigmoid
from Pipeline.Model.EvaluationMatrix import EvaluationMatrix

In [ ]:
config_file = '../../Dataset/JSON/full_model_configs.json'

model_configs = []

try:
    with open(config_file, 'r') as f:
        configs = json.load(f)
        for config in configs:
            if config["Activation"] == "sigmoid":
                config["Activation"] = sigmoid
        model_configs.extend(configs)
except FileNotFoundError:
    print(f"Error: {config_file} not found. Run ELM_Evaluation and Grid_Evaluation first.")

print(f"Loaded {len(model_configs)} configurations ready for testing.")

In [ ]:
gallstone_dataset = GallstoneDataSet()
gallstone_dataset.fetch_data_path_1()
features_size = gallstone_dataset.x_train_scaled.shape[1]

In [ ]:
x_test_scaled = gallstone_dataset.x_test_scaled
y_test = gallstone_dataset.y_test
x_train_scaled = gallstone_dataset.x_train_scaled
y_train = gallstone_dataset.y_train

In [ ]:
testing_results = []
for config in model_configs:
    print(f"Executing: Nodes : {config['Hidden_Nodes']} , lambda value : {config['Lambda_Value']} , activation function : {config['Activation'].__name__}")

    elm = ExtremeLearningMachine(
        features_size   = features_size,
        hidden_size     = config["Hidden_Nodes"],
        activation_function     = config["Activation"],
        regularization_lambda   = config["Lambda_Value"]
    )
    elm.initialize_random_weights(random_seed=42)


    elm.fit(x_train_scaled.values, y_train.values)
    y_pred = elm.predict(x_test_scaled.values)

    evaluation = EvaluationMatrix(y_test, y_pred)
    print(evaluation.get_report())

    metrics = evaluation.get_all_metrics()
    test_result = {
        "Model_Type"   : config.get('Model_Types', 'Unknown'), # Extract the strategy name
        "Hidden_Nodes" : config['Hidden_Nodes'],
        "Lambda_Value" : config['Lambda_Value'],
        "Activation"   : config['Activation'].__name__
    }
    test_result.update(metrics)
    testing_results.append(test_result)
    print("\n")

In [ ]:
final_model_result = pd.DataFrame(testing_results)
final_model_result